# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yuan05-afk/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

Contract for Refresh / Content Opportunity Scoring on the **starter CSV**. Warehouse queries
are deferred (no HF token in this environment); SETUP.md documents the access path.


## 1. Unit of analysis + time window

**Five plain-words answers:**

1. **One row means:** one pseudonymized content item (page) for one client.
2. **Table:** `data/raw/content_refresh_anonymized.csv` (starter). Warehouse later:
   `dim_content` + aggregated `fact_content_daily_performance` windows.
3. **Time window:** trailing 90-day metrics baked into the snapshot columns (`*_90d`,
   `*_last_30d`, `*_prev_30d`).
4. **Label / rank target:** decline proxy `is_declining_label` / rank by refresh priority score.
5. **Action:** editor opens the top of a ranked refresh queue.

Verification below uses starter grain counts (warehouse `IS TRUE` availability checks deferred).


In [1]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

ROOT = Path('.').resolve()
# Support running from repo root or work/notebooks
if (ROOT / 'data/raw/content_refresh_anonymized.csv').exists():
    DATA = ROOT / 'data/raw/content_refresh_anonymized.csv'
elif (ROOT.parents[1] / 'data/raw/content_refresh_anonymized.csv').exists():
    ROOT = ROOT.parents[1]
    DATA = ROOT / 'data/raw/content_refresh_anonymized.csv'
else:
    # Colab / nested
    for cand in [ROOT, *ROOT.parents]:
        if (cand / 'data/raw/content_refresh_anonymized.csv').exists():
            ROOT = cand
            DATA = cand / 'data/raw/content_refresh_anonymized.csv'
            break
    else:
        raise FileNotFoundError('data/raw/content_refresh_anonymized.csv')

WORK_OUT = ROOT / 'work' / 'outputs'
WORK_OUT.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(DATA)
print(f'Loaded {len(df):,} rows × {df.shape[1]} cols from {DATA.relative_to(ROOT)}')
print(f'Clients (pseudonymous): {df["client_id"].nunique()}')

print('Grain probe: duplicate content_id?')
dup = df['content_id'].duplicated().sum()
print('duplicate content_id rows:', int(dup))
print('rows:', len(df), '| clients:', df['client_id'].nunique())


Loaded 30,000 rows × 44 cols from data\raw\content_refresh_anonymized.csv
Clients (pseudonymous): 32
Grain probe: duplicate content_id?
duplicate content_id rows: 0
rows: 30000 | clients: 32


## 2. Fields: feature / label / context / excluded

| Bucket | Fields |
|---|---|
| **Features (safe)** | search_volume, competition, cpc, word/char counts, log impressions/clicks/sessions/AI sessions, days_with_*, content_age_days, days_since_last_update, ctr, avg_position, engagement_rate, scroll_rate, ai_traffic_pct + categoricals (competition_level, content_type, main_intent, age/freshness/word/impression/position tiers) |
| **Label** | is_declining_label (from trend_direction==down) |
| **Context only** | content_id, client_id (split/group keys, never features) |
| **Excluded** | trend_direction, trend_pct (label leakage); any raw URL/title/query/client name (privacy; not in starter) |


In [2]:
FEATURE_NUM = [
    'search_volume','competition','cpc','word_count','char_count','impressions_90d','clicks_90d',
    'sessions_90d','ai_sessions_90d','days_with_impressions','days_with_sessions','content_age_days',
    'days_since_last_update','ctr','avg_position','engagement_rate','scroll_rate','ai_traffic_pct'
]
EXCLUDED = ['trend_direction','trend_pct']
LABEL = 'is_declining_label'
print('Feature columns present:', sum(c in df.columns for c in FEATURE_NUM), '/', len(FEATURE_NUM))
print('Excluded present (must not train on):', [c for c in EXCLUDED if c in df.columns])


Feature columns present: 18 / 18
Excluded present (must not train on): ['trend_direction', 'trend_pct']


## 3. Verify it with queries (grain, counts, missing values, windows)

Three verification queries with outputs visible. Starter stand-in for warehouse `IS TRUE` checks.


In [3]:
# Query 1 — grain + counts
q1 = {
    'n_rows': int(len(df)),
    'n_content': int(df['content_id'].nunique()),
    'n_clients': int(df['client_id'].nunique()),
    'grain_ok': bool(df['content_id'].nunique() == len(df)),
}
print('Q1 grain:', json.dumps(q1))

# Query 2 — missingness on key features (by content_type awareness)
miss = df[FEATURE_NUM].isna().mean().sort_values(ascending=False).head(8)
print('\nQ2 top missing rates:')
print((miss*100).round(2).astype(str) + '%')

# Query 3 — window sanity: last30 vs prev30 presence + avg_position==0 gotcha
q3 = {
    'avg_position_zero_rows': int((df['avg_position']==0).sum()),
    'pct_avg_position_zero': round(float((df['avg_position']==0).mean())*100, 2),
    'impressions_last_30d_median': float(df['impressions_last_30d'].median()),
    'impressions_prev_30d_median': float(df['impressions_prev_30d'].median()),
}
print('\nQ3 windows/gotchas:', json.dumps(q3, indent=2))
print('Note: avg_position=0 means no data, not rank zero.')

# Five-feature frame with available-when
frame = pd.DataFrame([
    {'feature':'days_since_last_update','available_when':'always in starter','role':'freshness risk'},
    {'feature':'impressions_90d','available_when':'GSC history present','role':'visibility'},
    {'feature':'avg_position','available_when':'avg_position>0','role':'SERP context'},
    {'feature':'ctr','available_when':'impressions>0; rate is ×100 percent','role':'CTR gap'},
    {'feature':'word_count','available_when':'not missing for content_type','role':'depth gap'},
])
print('\nFive-feature frame:')
print(frame.to_string(index=False))
(WORK_OUT / 'w03_contract_checks.json').write_text(json.dumps({'q1':q1,'q3':q3}, indent=2))


Q1 grain: {"n_rows": 30000, "n_content": 30000, "n_clients": 32, "grain_ok": true}

Q2 top missing rates:
char_count       25.66%
word_count       25.66%
competition       8.23%
search_volume     8.23%
cpc               8.23%
scroll_rate       0.42%
clicks_90d         0.0%
sessions_90d       0.0%
dtype: str

Q3 windows/gotchas: {
  "avg_position_zero_rows": 1205,
  "pct_avg_position_zero": 4.02,
  "impressions_last_30d_median": 139.0,
  "impressions_prev_30d_median": 210.0
}
Note: avg_position=0 means no data, not rank zero.

Five-feature frame:
               feature                      available_when           role
days_since_last_update                   always in starter freshness risk
       impressions_90d                 GSC history present     visibility
          avg_position                      avg_position>0   SERP context
                   ctr impressions>0; rate is ×100 percent        CTR gap
            word_count        not missing for content_type      depth gap


273

## 4. Data limits

- Starter is a **snapshot**, not a full panel; warehouse daily facts unlock true past→future labels.
- Panel depth differs by client in the warehouse; one global calendar window would be wrong.
- Rate columns are ×100 percentages (`ctr=0.76` means 0.76%).
- `scroll_rate` / `ai_traffic_pct` can exceed 100 (cross-system numerators).
- Missingness tracks `content_type`; blind `fillna(0)` injects category signal.
- Without HF access this week, warehouse verification queries are **deferred** (see SETUP.md).


In [4]:
print('Limits acknowledged. Warehouse deferred: no HF_TOKEN in this run.')
print('Contract still complete on starter CSV with three verification queries above.')


Limits acknowledged. Warehouse deferred: no HF_TOKEN in this run.
Contract still complete on starter CSV with three verification queries above.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
